# Georgia Power 2025 IRP: prepare project PDFs

Download the original public-disclosure ZIP from [Georgia PSC filing 221233](https://psc.ga.gov/search/facts-document/?documentId=221233), extract its files, find Volume 3, and copy only the requested Georgia ITS Ten-Year Plan sections.

**This notebook does not parse project data.** It only reads headings and page labels to select PDF pages. Original ZIP and PDF files are reused without overwriting them.

Use the existing project `.venv` kernel. We reuse `requests`, `re`, and PyMuPDF (`fitz`). The one new module, Python's built-in `zipfile`, reads ZIP archives; it needs no installation. The manual alternative is File Explorer's Extract All. We flatten the archive into a single folder because its nested paths exceed Windows' path-length limit here; filenames and file contents are preserved, and duplicate names stop extraction.

In [8]:
import requests
import zipfile
import fitz  # PyMuPDF
import re

from gridlock.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR

SOURCE_PAGE = "https://psc.ga.gov/search/facts-document/?documentId=221233"
ZIP_URL = "https://services.psc.ga.gov/api/v1/External/Public/Get/Document/DownloadFile/221233/102406"
RAW_DIR = RAW_DATA_DIR / "georgia_power" / "2025_irp"
ZIP_PATH = RAW_DIR / "2025_irp_public_disclosure.zip"
EXTRACT_DIR = RAW_DIR / "files"
OUTPUT_DIR = RAW_DATA_DIR / "georgia_power" /  "2025_irp" / "selected"

RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download once
If the original ZIP exists, reuse it. `"xb"` creates a new file and refuses to overwrite an existing one. The HTTP request must succeed and return ZIP bytes before anything is saved.

In [9]:
if ZIP_PATH.exists():
    print("Reusing original ZIP:", ZIP_PATH)
else:
    response = requests.get(ZIP_URL, timeout=180)
    response.raise_for_status()
    if not response.content.startswith(b"PK"):
        raise ValueError("The download did not return a ZIP file.")
    with ZIP_PATH.open("xb") as file:
        file.write(response.content)
    print("Downloaded:", ZIP_PATH)

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError("The saved file is not a valid ZIP archive.")

Reusing original ZIP: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\2025_irp_public_disclosure.zip


## 2. Extract the files and locate Volume 3
Open the ZIP in read mode. Check for duplicate filenames before extracting, and skip files that already exist. Compare existing files with their archive contents so an incomplete or changed extraction cannot silently be reused.

In [10]:
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
volume_paths = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    filenames = []
    for item in archive.infolist():
        if not item.is_dir():
            filename = item.filename.split("/")[-1]
            if filename.lower() in filenames:
                raise ValueError("Duplicate filename in ZIP: " + filename)
            filenames.append(filename.lower())

    for item in archive.infolist():
        if item.is_dir():
            continue
        filename = item.filename.split("/")[-1]
        file_path = EXTRACT_DIR / filename
        with archive.open(item) as source:
            contents = source.read()
        if file_path.exists():
            if file_path.read_bytes() != contents:
                raise ValueError("Existing extracted file differs from ZIP: " + filename)
        else:
            with file_path.open("xb") as file:
                file.write(contents)
        if filename.lower() == "2025 irp volume 3 public disclosure.pdf":
            volume_paths.append(file_path)

if len(volume_paths) != 1:
    raise ValueError("Expected exactly one 2025 IRP Volume 3 PUBLIC DISCLOSURE PDF.")
VOLUME3_PATH = volume_paths[0]
print("Extracted files:", len(filenames))
print("Volume 3:", VOLUME3_PATH)

Extracted files: 57
Volume 3: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\files\2025 IRP Volume 3 PUBLIC DISCLOSURE.pdf


## 3. Locate the Georgia ITS Ten-Year Plan
PyMuPDF's `get_toc()` reads PDF bookmarks. Each bookmark supplies a title and its actual PDF page number. The bookmark's title contains old page references, so we use its destination, then verify the plan heading and its printed `Page 1 of ...` label.

PDF page numbers shown to readers start at 1; Python page indexes start at 0.

In [11]:
with fitz.open(VOLUME3_PATH) as document:
    plan_starts = []
    for level, title, page_number in document.get_toc():
        if "GA ITS Ten Year Plan" in title:
            plan_starts.append(page_number - 1)
    if len(plan_starts) != 1:
        raise ValueError("Could not identify one Georgia ITS Ten-Year Plan bookmark.")
    plan_start = plan_starts[0]
    first_page_text = document[plan_start].get_text("text")
    if "GA ITS Ten-Year Plan" not in first_page_text:
        raise ValueError("The bookmark does not point to the expected plan.")
    page_label = re.search(r"Page\s+1\s+of\s+(\d+)", first_page_text)
    if not page_label:
        raise ValueError("The plan's first printed page label was not found.")
    plan_page_count = int(page_label.group(1))
    plan_texts = []
    for index in range(plan_start, plan_start + plan_page_count):
        plan_texts.append(document[index].get_text("text"))

print("Plan starts at Volume 3 PDF page:", plan_start + 1)
print("Pages in the plan:", plan_page_count)

Plan starts at Volume 3 PDF page: 171
Pages in the plan: 304


## 4. Read the plan's contents and printed page labels
The contents lines contain a heading, dotted leaders, and a printed page number. We read those entries from the opening contents pages and build a lookup from printed page labels to actual PDF indexes. No section boundaries are hardcoded to the approximate page numbers in the request.

In [12]:
toc_entries = []
toc_pdf_pages = []
printed_pages = {}

for index, text in enumerate(plan_texts):
    label = re.search(r"Page\s+(\d+)\s+of\s+(\d+)", text)
    if not label or int(label.group(2)) != plan_page_count:
        raise ValueError("Missing or unexpected page label at plan index " + str(index))
    printed_page = int(label.group(1))
    if printed_page in printed_pages:
        raise ValueError("Duplicate printed page label: " + str(printed_page))
    printed_pages[printed_page] = plan_start + index

    # The plan's contents occupies its opening pages, before the body sections.
    if index < 5:
        found_entries = False
        for line in text.splitlines():
            entry = re.match(r"^\s*(.*?)\.{2,}\s*(\d+)\s*$", line)
            if entry:
                toc_entries.append((entry.group(1).strip(), int(entry.group(2))))
                found_entries = True
        if found_entries:
            toc_pdf_pages.append(plan_start + index + 1)

if not toc_entries:
    raise ValueError("No table-of-contents entries were found.")
if sorted(printed_pages) != list(range(1, plan_page_count + 1)):
    raise ValueError("The printed plan page labels are not complete.")

print("Table of contents found on Volume 3 PDF pages:", toc_pdf_pages)
for page_number in toc_pdf_pages:
    print(plan_texts[page_number - 1 - plan_start])

Table of contents found on Volume 3 PDF pages: [172, 173]
CRITICAL ENERGY INFRASTRUCTURE INFORMATION - CONFIDENTIAL. This data is confidential CEII and is subject to Regulation by CFR Sec. 388.113.  Recipient should 
be aware that disclosure of this material and its contents shall be handled in accordance with CEII procedures. Any and all duplications of this data must contain this 
notification. This document contains non-public transmission information and in accordance with FERC policy, should not be disclosed to Marketing Function 
employees. 
 
 
 
2024 GA ITS Ten-Year Plan (2025-2034)  
 
                 Page 2 of 304 
 
 
CONTENTS 
I. 
GA ITS EXECUTIVE SUMMARY ..................................................................................... 4 
A. Georgia ITS 10 Year Expansion Plan Projects List ........................................................................... 7 
B. Cancelled Projects List – Removed from the Current 10 Year Expansion Plan ..........................

## 5. Verify the six sections and their end boundaries
For each requested heading, find its contents entry and the next section's entry. Stop just before that next section. Verify both headings on the actual pages, and check that the intervening printed pages are consecutive.

The following sections are excluded: cancelled/completed projects, planning methods, generation-unit details, and fleet-transition tables. Whole pages are copied, preserving their layout and existing notices.

In [13]:
sections = [
    ("Georgia ITS 10 Year Expansion Plan Projects List", "Cancelled Projects List", "01_expansion_plan_projects.pdf"),
    ("Stability Project Details", "Short Circuit Project Details", "02_stability_projects.pdf"),
    ("Short Circuit Project Details", "Interface Transfer Capability Project Details", "03_short_circuit_projects.pdf"),
    ("Interface Transfer Capability Project Details", "Steady State Project Details", "04_interface_transfer_projects.pdf"),
    ("Steady State Project Details", "Expansion Generation Units Details", "05_steady_state_projects.pdf"),
    ("Strategic Projects", "Fleet Transition Tables", "06_strategic_projects.pdf"),
]

ranges = []
for name, next_name, filename in sections:
    boundaries = []
    for heading in [name, next_name]:
        matches = []
        for toc_name, printed_page in toc_entries:
            if heading in toc_name:
                matches.append(printed_page)
        if len(matches) != 1:
            raise ValueError("Expected one contents entry for: " + heading)
        printed_page = matches[0]
        pdf_index = printed_pages[printed_page]
        page_text = " ".join(plan_texts[pdf_index - plan_start].split())
        if heading not in page_text:
            raise ValueError("Heading was not verified on its page: " + heading)
        boundaries.append(printed_page)

    first_page = boundaries[0]
    last_page = boundaries[1] - 1
    if last_page < first_page:
        raise ValueError("Invalid section range: " + name)
    start_index = printed_pages[first_page]
    end_index = printed_pages[last_page]
    for page in range(first_page, last_page + 1):
        if printed_pages[page] != start_index + page - first_page:
            raise ValueError("Nonconsecutive pages in: " + name)
    ranges.append({
        "section": name,
        "plan_first": first_page,
        "plan_last": last_page,
        "start_index": start_index,
        "end_index": end_index,
        "output_path": OUTPUT_DIR / filename,
    })
    print(name, "| Plan pages:", first_page, "-", last_page,
          "| Volume 3 PDF pages:", start_index + 1, "-", end_index + 1)

Georgia ITS 10 Year Expansion Plan Projects List | Plan pages: 7 - 20 | Volume 3 PDF pages: 177 - 190
Stability Project Details | Plan pages: 43 - 53 | Volume 3 PDF pages: 213 - 223
Short Circuit Project Details | Plan pages: 54 - 55 | Volume 3 PDF pages: 224 - 225
Interface Transfer Capability Project Details | Plan pages: 56 - 59 | Volume 3 PDF pages: 226 - 229
Steady State Project Details | Plan pages: 60 - 255 | Volume 3 PDF pages: 230 - 425
Strategic Projects | Plan pages: 272 - 297 | Volume 3 PDF pages: 442 - 467


## 6. Create the working PDFs and print the summary
`insert_pdf()` copies a page range into a new PDF. Only working files under `data/processed/` are regenerated on reruns; the original ZIP and extracted files under `data/raw/` are never overwritten.

In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
total_pages = 0
summary_lines = [
    "Official source: " + SOURCE_PAGE,
    "Downloaded ZIP: " + str(ZIP_PATH),
    "Volume 3 PDF: " + str(VOLUME3_PATH),
    "Table of contents PDF pages: " + str(toc_pdf_pages),
    "",
]

with fitz.open(VOLUME3_PATH) as original:
    for section in ranges:
        with fitz.open() as working:
            working.insert_pdf(original, from_page=section["start_index"], to_page=section["end_index"])
            working.save(section["output_path"])
        page_count = section["end_index"] - section["start_index"] + 1
        with fitz.open(section["output_path"]) as saved:
            if len(saved) != page_count:
                raise ValueError("Unexpected output page count.")
        total_pages += page_count
        summary_lines.append(section["section"])
        summary_lines.append(f"  Plan pages: {section['plan_first']}-{section['plan_last']}")
        summary_lines.append(f"  Volume 3 PDF pages: {section['start_index'] + 1}-{section['end_index'] + 1}")
        summary_lines.append(f"  Pages extracted: {page_count}")
        summary_lines.append("  Output PDF: " + str(section["output_path"]))
        summary_lines.append("")

summary_lines.append("Total pages extracted: " + str(total_pages))
summary = "\n".join(summary_lines)
print(summary)
with (OUTPUT_DIR / "extraction_summary.txt").open("w", encoding="utf-8") as file:
    file.write(summary + "\n")

Official source: https://psc.ga.gov/search/facts-document/?documentId=221233
Downloaded ZIP: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\2025_irp_public_disclosure.zip
Volume 3 PDF: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\files\2025 IRP Volume 3 PUBLIC DISCLOSURE.pdf
Table of contents PDF pages: [172, 173]

Georgia ITS 10 Year Expansion Plan Projects List
  Plan pages: 7-20
  Volume 3 PDF pages: 177-190
  Pages extracted: 14
  Output PDF: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\selected\01_expansion_plan_projects.pdf

Stability Project Details
  Plan pages: 43-53
  Volume 3 PDF pages: 213-223
  Pages extracted: 11
  Output PDF: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\selected\02_stability_projects.pdf

Short Circuit Project Details
  Plan pages: 54-55
  Volume 3 PDF pages: 224-225
  Pages extr